### Setup

In [1]:
from pathlib import Path
import sys

import torch
import torch.nn as nn


def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            (candidate / "main.py").exists()
            and
            (candidate / "utils").exists()
            and
            (candidate / "thesis").exists()
        ):
            return candidate

    raise RuntimeError(
        "PyCIL project root not found."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


from thesis.backbones.resnet_attention import (
    resnet18_remote,
    resnet34_remote,
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "CUDA:",
    torch.cuda.is_available(),
)

Project root: C:\Skripsi\Project\PyCIL
PyTorch: 2.6.0+cu126
CUDA: True


### Validate standard stem

In [2]:
args = {
    "attention_type": "none"
}

model18 = resnet18_remote(
    pretrained=False,
    args=args,
)

print(model18.conv1)
print(model18.maxpool)

assert model18.conv1.kernel_size == (
    7,
    7,
)

assert model18.conv1.stride == (
    2,
    2,
)

assert model18.maxpool.kernel_size == 3
assert model18.maxpool.stride == 2

print(
    "Standard ResNet stem: PASSED"
)

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
Standard ResNet stem: PASSED


### Validate parameter count

In [3]:
model18 = resnet18_remote(
    pretrained=False,
    args=args,
)

model34 = resnet34_remote(
    pretrained=False,
    args=args,
)


def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


params18 = count_parameters(
    model18
)

params34 = count_parameters(
    model34
)

print(
    "ResNet-18 backbone params:",
    f"{params18:,}",
)

print(
    "ResNet-34 backbone params:",
    f"{params34:,}",
)


assert params18 == 11_176_512
assert params34 == 21_284_672

print(
    "Parameter count: PASSED"
)

ResNet-18 backbone params: 11,176,512
ResNet-34 backbone params: 21,284,672
Parameter count: PASSED


### Validate basic block count

In [4]:
from thesis.backbones.resnet_attention import (
    BasicBlock,
)


def count_basic_blocks(model):
    return sum(
        isinstance(module, BasicBlock)
        for module in model.modules()
    )


print(
    "ResNet-18 blocks:",
    count_basic_blocks(model18),
)

print(
    "ResNet-34 blocks:",
    count_basic_blocks(model34),
)


assert (
    count_basic_blocks(model18)
    == 8
)

assert (
    count_basic_blocks(model34)
    == 16
)

print(
    "BasicBlock count: PASSED"
)

ResNet-18 blocks: 8
ResNet-34 blocks: 16
BasicBlock count: PASSED


### Validate resnet with no attention

In [5]:
identity_count_18 = sum(
    isinstance(module, nn.Identity)
    for module in model18.modules()
)

identity_count_34 = sum(
    isinstance(module, nn.Identity)
    for module in model34.modules()
)

print(
    "ResNet-18 Identity attention:",
    identity_count_18,
)

print(
    "ResNet-34 Identity attention:",
    identity_count_34,
)

assert identity_count_18 == 8
assert identity_count_34 == 16

print(
    "No-attention baseline: PASSED"
)

ResNet-18 Identity attention: 8
ResNet-34 Identity attention: 16
No-attention baseline: PASSED


### Forward Pass NWPU 256x256

In [6]:
model18.eval()

x_nwpu = torch.randn(
    1,
    3,
    256,
    256,
)

with torch.no_grad():
    output = model18(
        x_nwpu
    )


print(
    "Features:",
    output["features"].shape,
)

for i, fmap in enumerate(
    output["fmaps"],
    start=1,
):
    print(
        f"Layer {i}:",
        fmap.shape,
    )

Features: torch.Size([1, 512])
Layer 1: torch.Size([1, 64, 64, 64])
Layer 2: torch.Size([1, 128, 32, 32])
Layer 3: torch.Size([1, 256, 16, 16])
Layer 4: torch.Size([1, 512, 8, 8])


In [7]:
assert output[
    "features"
].shape == (1, 512)

expected_nwpu = [
    (1, 64, 64, 64),
    (1, 128, 32, 32),
    (1, 256, 16, 16),
    (1, 512, 8, 8),
]

assert [
    tuple(x.shape)
    for x in output["fmaps"]
] == expected_nwpu

print(
    "NWPU forward: PASSED"
)

NWPU forward: PASSED


### Forward pass AID 600x600

In [8]:
model18.eval()

x_aid = torch.randn(
    1,
    3,
    600,
    600,
)

with torch.no_grad():
    output = model18(
        x_aid
    )


print(
    "Features:",
    output["features"].shape,
)

for i, fmap in enumerate(
    output["fmaps"],
    start=1,
):
    print(
        f"Layer {i}:",
        fmap.shape,
    )

Features: torch.Size([1, 512])
Layer 1: torch.Size([1, 64, 150, 150])
Layer 2: torch.Size([1, 128, 75, 75])
Layer 3: torch.Size([1, 256, 38, 38])
Layer 4: torch.Size([1, 512, 19, 19])


In [9]:
expected_aid = [
    (1, 64, 150, 150),
    (1, 128, 75, 75),
    (1, 256, 38, 38),
    (1, 512, 19, 19),
]

assert [
    tuple(x.shape)
    for x in output["fmaps"]
] == expected_aid

assert output[
    "features"
].shape == (1, 512)

print(
    "AID forward: PASSED"
)

AID forward: PASSED


### Validate ResNet-34

In [10]:
model34.eval()

for image_size in [
    256,
    600,
]:
    x = torch.randn(
        1,
        3,
        image_size,
        image_size,
    )

    with torch.no_grad():
        output = model34(x)

    print(
        f"Input {image_size}x{image_size}"
    )

    print(
        "Feature:",
        output[
            "features"
        ].shape,
    )

    print(
        "Final fmap:",
        output[
            "fmaps"
        ][-1].shape,
    )

    assert (
        output["features"].shape
        == (1, 512)
    )


print(
    "ResNet-34 forward: PASSED"
)

Input 256x256
Feature: torch.Size([1, 512])
Final fmap: torch.Size([1, 512, 8, 8])
Input 600x600
Feature: torch.Size([1, 512])
Final fmap: torch.Size([1, 512, 19, 19])
ResNet-34 forward: PASSED


### Integrate with incremental net

In [11]:
from utils.inc_net import (
    IncrementalNet,
)


inc_args = {
    "convnet_type":
        "remote_resnet18",

    "attention_type":
        "none",
}


network = IncrementalNet(
    inc_args,
    pretrained=False,
)

network.update_fc(
    nb_classes=14
)

network.eval()

x = torch.randn(
    1,
    3,
    600,
    600,
)

with torch.no_grad():
    output = network(x)


print(
    "Features:",
    output["features"].shape,
)

print(
    "Logits:",
    output["logits"].shape,
)

print(
    "Fmaps:",
    [
        tuple(f.shape)
        for f in output["fmaps"]
    ],
)


assert output[
    "features"
].shape == (1, 512)

assert output[
    "logits"
].shape == (1, 14)

print(
    "IncrementalNet integration: PASSED"
)

Features: torch.Size([1, 512])
Logits: torch.Size([1, 14])
Fmaps: [(1, 64, 150, 150), (1, 128, 75, 75), (1, 256, 38, 38), (1, 512, 19, 19)]
IncrementalNet integration: PASSED


### Backward pass GPU

In [12]:
import torch.nn.functional as F


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


network = IncrementalNet(
    inc_args,
    pretrained=False,
)

network.update_fc(
    nb_classes=14
)

network = network.to(
    device
)

network.train()


x = torch.randn(
    1,
    3,
    600,
    600,
    device=device,
)

target = torch.tensor(
    [3],
    dtype=torch.long,
    device=device,
)


output = network(x)

loss = F.cross_entropy(
    output["logits"],
    target,
)

network.zero_grad()

loss.backward()


has_gradient = any(
    parameter.grad is not None
    for parameter
    in network.parameters()
    if parameter.requires_grad
)


print(
    "Device:",
    device,
)

print(
    "Loss:",
    loss.item(),
)

print(
    "Gradient exists:",
    has_gradient,
)

assert has_gradient

print(
    "Backward pass: PASSED"
)


del network
del x
del target
del output

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cuda
Loss: 2.209366798400879
Gradient exists: True
Backward pass: PASSED


### Validate last_conv

In [13]:
model18 = resnet18_remote(
    pretrained=False,
    args=args,
)

print(
    model18.last_conv
)

assert (
    model18.last_conv
    is model18.layer4[-1].conv2
)

print(
    "last_conv compatibility: PASSED"
)

Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
last_conv compatibility: PASSED


### Validate ECA module

In [14]:
# import sys
# import os

# # Add the parent directory (PyCIL) to the Python path
# sys.path.append(os.path.abspath('..'))

from thesis.attention.eca import (
    EfficientChannelAttention,
)


channels = [
    64,
    128,
    256,
    512,
]


for c in channels:

    module = (
        EfficientChannelAttention(
            channels=c
        )
    )

    print(
        f"C={c}: "
        f"k={module.kernel_size}"
    )

C=64: k=3
C=128: k=5
C=256: k=5
C=512: k=5


In [15]:
expected = {
    64: 3,
    128: 5,
    256: 5,
    512: 5,
}

for c, expected_k in (
    expected.items()
):
    module = (
        EfficientChannelAttention(
            channels=c
        )
    )

    assert (
        module.kernel_size
        == expected_k
    )

print(
    "ECA adaptive kernel: PASSED"
)

ECA adaptive kernel: PASSED


### ECA shape preservation test

In [16]:
x = torch.randn(
    2,
    128,
    32,
    32,
)

eca = (
    EfficientChannelAttention(
        channels=128
    )
)

with torch.no_grad():
    y = eca(x)


print(
    "Input :",
    x.shape,
)

print(
    "Output:",
    y.shape,
)


assert y.shape == x.shape

print(
    "ECA shape preservation: PASSED"
)

Input : torch.Size([2, 128, 32, 32])
Output: torch.Size([2, 128, 32, 32])
ECA shape preservation: PASSED


### Validate ECA module count

In [17]:
eca_args = {
    "attention_type": "eca",
    "eca_gamma": 2.0,
    "eca_b": 1.0,
}


eca18 = resnet18_remote(
    pretrained=False,
    args=eca_args,
)

eca34 = resnet34_remote(
    pretrained=False,
    args=eca_args,
)


eca_count_18 = sum(
    isinstance(
        module,
        EfficientChannelAttention,
    )
    for module in eca18.modules()
)

eca_count_34 = sum(
    isinstance(
        module,
        EfficientChannelAttention,
    )
    for module in eca34.modules()
)


print(
    "ResNet-18 ECA modules:",
    eca_count_18,
)

print(
    "ResNet-34 ECA modules:",
    eca_count_34,
)


assert eca_count_18 == 8
assert eca_count_34 == 16

print(
    "ECA per BasicBlock: PASSED"
)

ResNet-18 ECA modules: 8
ResNet-34 ECA modules: 16
ECA per BasicBlock: PASSED


### Check kernel in every block

In [18]:
def get_eca_kernels(model):

    kernels = []

    for module in model.modules():

        if isinstance(
            module,
            EfficientChannelAttention,
        ):
            kernels.append(
                module.kernel_size
            )

    return kernels


print(
    "ResNet-18:",
    get_eca_kernels(
        eca18
    ),
)

print(
    "ResNet-34:",
    get_eca_kernels(
        eca34
    ),
)

ResNet-18: [3, 3, 5, 5, 5, 5, 5, 5]
ResNet-34: [3, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]


### Check ECA overhead

In [19]:
baseline18 = resnet18_remote(
    pretrained=False,
    args={
        "attention_type":
            "none"
    },
)

baseline34 = resnet34_remote(
    pretrained=False,
    args={
        "attention_type":
            "none"
    },
)


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


base18_params = count_params(
    baseline18
)

eca18_params = count_params(
    eca18
)

base34_params = count_params(
    baseline34
)

eca34_params = count_params(
    eca34
)


print(
    "R18 baseline:",
    f"{base18_params:,}",
)

print(
    "R18 ECA     :",
    f"{eca18_params:,}",
)

print(
    "R18 added   :",
    eca18_params
    - base18_params,
)

print()

print(
    "R34 baseline:",
    f"{base34_params:,}",
)

print(
    "R34 ECA     :",
    f"{eca34_params:,}",
)

print(
    "R34 added   :",
    eca34_params
    - base34_params,
)

R18 baseline: 11,176,512
R18 ECA     : 11,176,548
R18 added   : 36

R34 baseline: 21,284,672
R34 ECA     : 21,284,746
R34 added   : 74


### Forward AID and NWPU

In [20]:
eca18.eval()

for size in [
    256,
    600,
]:

    x = torch.randn(
        1,
        3,
        size,
        size,
    )

    with torch.no_grad():
        output = eca18(x)

    print(
        f"{size}x{size}:"
    )

    print(
        " feature =",
        output[
            "features"
        ].shape,
    )

    print(
        " final fmap =",
        output[
            "fmaps"
        ][-1].shape,
    )

    assert (
        output[
            "features"
        ].shape
        == (1, 512)
    )


print(
    "ECA forward: PASSED"
)

256x256:
 feature = torch.Size([1, 512])
 final fmap = torch.Size([1, 512, 8, 8])
600x600:
 feature = torch.Size([1, 512])
 final fmap = torch.Size([1, 512, 19, 19])
ECA forward: PASSED


### Gradiant test until it reaches ECA?

In [21]:
eca18 = resnet18_remote(
    pretrained=False,
    args=eca_args,
)

eca18.train()

x = torch.randn(
    1,
    3,
    256,
    256,
)

output = eca18(x)

loss = (
    output[
        "features"
    ].mean()
)

eca18.zero_grad()

loss.backward()


eca_gradients = []

for name, parameter in (
    eca18.named_parameters()
):

    if (
        ".attention.conv."
        in name
    ):
        eca_gradients.append(
            (
                name,
                parameter.grad
                is not None,
            )
        )


for item in eca_gradients:
    print(item)


assert len(
    eca_gradients
) == 8

assert all(
    has_grad
    for _, has_grad
    in eca_gradients
)

print(
    "ECA gradient flow: PASSED"
)

('layer1.0.attention.conv.weight', True)
('layer1.1.attention.conv.weight', True)
('layer2.0.attention.conv.weight', True)
('layer2.1.attention.conv.weight', True)
('layer3.0.attention.conv.weight', True)
('layer3.1.attention.conv.weight', True)
('layer4.0.attention.conv.weight', True)
('layer4.1.attention.conv.weight', True)
ECA gradient flow: PASSED


### Test through IncrementalNet

In [22]:
from utils.inc_net import (
    IncrementalNet,
)


inc_eca_args = {
    "convnet_type":
        "remote_resnet18",

    "attention_type":
        "eca",

    "eca_gamma":
        2.0,

    "eca_b":
        1.0,
}


network = IncrementalNet(
    inc_eca_args,
    pretrained=False,
)

network.update_fc(
    nb_classes=14
)

network.eval()


x = torch.randn(
    1,
    3,
    600,
    600,
)


with torch.no_grad():
    output = network(x)


print(
    "Feature:",
    output[
        "features"
    ].shape,
)

print(
    "Logits:",
    output[
        "logits"
    ].shape,
)


assert (
    output[
        "features"
    ].shape
    == (1, 512)
)

assert (
    output[
        "logits"
    ].shape
    == (1, 14)
)


print(
    "ECA IncrementalNet: PASSED"
)

Feature: torch.Size([1, 512])
Logits: torch.Size([1, 14])
ECA IncrementalNet: PASSED


### Tes CBAM

In [23]:
from thesis.attention.cbam import (
    CBAM,
    ChannelAttention,
    SpatialAttention,
)


x = torch.randn(
    2,
    128,
    32,
    32,
)

cbam = CBAM(
    channels=128,
)

with torch.no_grad():
    y = cbam(x)


print(
    "Input :",
    x.shape,
)

print(
    "Output:",
    y.shape,
)


assert y.shape == x.shape

print(
    "CBAM shape preservation: PASSED"
)

Input : torch.Size([2, 128, 32, 32])
Output: torch.Size([2, 128, 32, 32])
CBAM shape preservation: PASSED


### Validate CBAM configuration

In [24]:
cbam = CBAM(
    channels=128,
    reduction_ratio=16,
    spatial_kernel_size=7,
)


print(
    "Channel reduction:",
    cbam.channel_attention
        .reduction_ratio,
)

print(
    "Spatial kernel:",
    cbam.spatial_attention
        .kernel_size,
)

assert (cbam.channel_attention.reduction_ratio == 16)
assert (cbam.spatial_attention.kernel_size == 7)

print("CBAM configuration: PASSED")

Channel reduction: 16
Spatial kernel: 7
CBAM configuration: PASSED


### Validate CBAM count

In [25]:
cbam_args = {
    "attention_type":
        "cbam",

    "cbam_reduction":
        16,

    "cbam_spatial_kernel":
        7,
}


cbam18 = resnet18_remote(
    pretrained=False,
    args=cbam_args,
)

cbam34 = resnet34_remote(
    pretrained=False,
    args=cbam_args,
)


cbam_count_18 = sum(
    isinstance(
        module,
        CBAM,
    )
    for module
    in cbam18.modules()
)

cbam_count_34 = sum(
    isinstance(
        module,
        CBAM,
    )
    for module
    in cbam34.modules()
)


print(
    "ResNet-18 CBAM modules:",
    cbam_count_18,
)

print(
    "ResNet-34 CBAM modules:",
    cbam_count_34,
)


assert cbam_count_18 == 8
assert cbam_count_34 == 16

print(
    "CBAM per BasicBlock: PASSED"
)

ResNet-18 CBAM modules: 8
ResNet-34 CBAM modules: 16
CBAM per BasicBlock: PASSED


### Verify sequential structure

In [26]:
for name, module in (
    cbam18.named_modules()
):

    if isinstance(
        module,
        CBAM,
    ):
        print(
            "First CBAM:",
            name,
        )

        print(
            "Channel:",
            type(
                module.channel_attention
            ).__name__,
        )

        print(
            "Spatial:",
            type(
                module.spatial_attention
            ).__name__,
        )

        break

First CBAM: layer1.0.attention
Channel: ChannelAttention
Spatial: SpatialAttention


### Parameter overhead CBAM

In [27]:
baseline18 = resnet18_remote(
    pretrained=False,
    args={
        "attention_type":
            "none"
    },
)

baseline34 = resnet34_remote(
    pretrained=False,
    args={
        "attention_type":
            "none"
    },
)


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


base18_params = count_params(
    baseline18
)

cbam18_params = count_params(
    cbam18
)

base34_params = count_params(
    baseline34
)

cbam34_params = count_params(
    cbam34
)


print(
    "R18 baseline:",
    f"{base18_params:,}",
)

print(
    "R18 CBAM    :",
    f"{cbam18_params:,}",
)

print(
    "R18 added   :",
    cbam18_params
    - base18_params,
)

print()

print(
    "R34 baseline:",
    f"{base34_params:,}",
)

print(
    "R34 CBAM    :",
    f"{cbam34_params:,}",
)

print(
    "R34 added   :",
    cbam34_params
    - base34_params,
)

R18 baseline: 11,176,512
R18 CBAM    : 11,266,392
R18 added   : 89880

R34 baseline: 21,284,672
R34 CBAM    : 21,447,468
R34 added   : 162796


### Forward AID and NWPU

In [28]:
cbam18.eval()


for size in [
    256,
    600,
]:

    x = torch.randn(
        1,
        3,
        size,
        size,
    )

    with torch.no_grad():
        output = cbam18(x)

    print(
        f"{size}x{size}:"
    )

    print(
        " feature =",
        output[
            "features"
        ].shape,
    )

    print(
        " final fmap =",
        output[
            "fmaps"
        ][-1].shape,
    )

    assert (
        output[
            "features"
        ].shape
        == (1, 512)
    )


print(
    "CBAM forward: PASSED"
)

256x256:
 feature = torch.Size([1, 512])
 final fmap = torch.Size([1, 512, 8, 8])
600x600:
 feature = torch.Size([1, 512])
 final fmap = torch.Size([1, 512, 19, 19])
CBAM forward: PASSED


### Test channel + spatial gradient

In [29]:
cbam18 = resnet18_remote(
    pretrained=False,
    args=cbam_args,
)

cbam18.train()


x = torch.randn(
    1,
    3,
    256,
    256,
)


output = cbam18(x)

loss = (
    output[
        "features"
    ].mean()
)

cbam18.zero_grad()

loss.backward()


channel_grad_count = 0
spatial_grad_count = 0


for name, parameter in (
    cbam18.named_parameters()
):

    if (
        ".attention."
        "channel_attention."
        in name
    ):
        if parameter.grad is not None:
            channel_grad_count += 1

    if (
        ".attention."
        "spatial_attention."
        in name
    ):
        if parameter.grad is not None:
            spatial_grad_count += 1


print(
    "Channel params with gradient:",
    channel_grad_count,
)

print(
    "Spatial params with gradient:",
    spatial_grad_count,
)


assert channel_grad_count > 0
assert spatial_grad_count > 0

print(
    "CBAM gradient flow: PASSED"
)

Channel params with gradient: 32
Spatial params with gradient: 24
CBAM gradient flow: PASSED


### Validate ProCBAM

In [30]:
from thesis.attention.procbam import (
    ProCBAM,
    SqueezeExcitation,
)


x = torch.randn(
    2,
    128,
    32,
    32,
)

procbam = ProCBAM(
    channels=128,
)

with torch.no_grad():
    y = procbam(x)


print(
    "Input :",
    x.shape,
)

print(
    "Output:",
    y.shape,
)

assert y.shape == x.shape

print(
    "ProCBAM shape preservation: PASSED"
)


Input : torch.Size([2, 128, 32, 32])
Output: torch.Size([2, 128, 32, 32])
ProCBAM shape preservation: PASSED


### Validate parallel computation

In [31]:
procbam.eval()

x = torch.randn(
    2,
    128,
    32,
    32,
)


with torch.no_grad():

    direct_output = (
        procbam(x)
    )

    channel_output = (
        procbam
        .channel_attention(x)
    )

    spatial_output = (
        procbam
        .spatial_attention(x)
    )

    fused = (
        channel_output
        + spatial_output
    )

    manual_output = (
        procbam
        .squeeze_excitation(
            fused
        )
    )


print(
    "Direct shape:",
    direct_output.shape,
)

print(
    "Manual shape:",
    manual_output.shape,
)


is_equal = torch.allclose(
    direct_output,
    manual_output,
    atol=1e-6,
)


print(
    "Parallel computation equal:",
    is_equal,
)

assert is_equal

print(
    "ProCBAM parallel structure: PASSED"
)

Direct shape: torch.Size([2, 128, 32, 32])
Manual shape: torch.Size([2, 128, 32, 32])
Parallel computation equal: True
ProCBAM parallel structure: PASSED


### Validate configuration

In [32]:
procbam = ProCBAM(
    channels=128,
    reduction_ratio=16,
    spatial_kernel_size=7,
    se_reduction_ratio=16,
)


print(
    "CAM reduction:",
    procbam.reduction_ratio,
)

print(
    "SAM kernel:",
    procbam.spatial_kernel_size,
)

print(
    "SE reduction:",
    procbam.se_reduction_ratio,
)


assert (
    procbam.reduction_ratio
    == 16
)

assert (
    procbam.spatial_kernel_size
    == 7
)

assert (
    procbam.se_reduction_ratio
    == 16
)


print(
    "ProCBAM configuration: PASSED"
)

CAM reduction: 16
SAM kernel: 7
SE reduction: 16
ProCBAM configuration: PASSED


### Validate ProCBAM module count in each ResNet model

In [33]:
procbam_args = {
    "attention_type": "procbam",

    "procbam_reduction":
        16,

    "procbam_spatial_kernel":
        7,

    "procbam_se_reduction":
        16,
}


procbam18 = resnet18_remote(
    pretrained=False,
    args=procbam_args,
)

procbam34 = resnet34_remote(
    pretrained=False,
    args=procbam_args,
)


count18 = sum(
    isinstance(
        module,
        ProCBAM,
    )
    for module
    in procbam18.modules()
)

count34 = sum(
    isinstance(
        module,
        ProCBAM,
    )
    for module
    in procbam34.modules()
)


print(
    "ResNet-18 ProCBAM modules:",
    count18,
)

print(
    "ResNet-34 ProCBAM modules:",
    count34,
)


assert count18 == 8
assert count34 == 16

print(
    "ProCBAM per BasicBlock: PASSED"
)

ResNet-18 ProCBAM modules: 8
ResNet-34 ProCBAM modules: 16
ProCBAM per BasicBlock: PASSED


### Parameter overhead from ProCBAM

In [34]:
baseline18 = resnet18_remote(
    pretrained=False,
    args={
        "attention_type":
            "none"
    },
)

baseline34 = resnet34_remote(
    pretrained=False,
    args={
        "attention_type":
            "none"
    },
)


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


base18 = count_params(
    baseline18
)

pro18 = count_params(
    procbam18
)

base34 = count_params(
    baseline34
)

pro34 = count_params(
    procbam34
)


print(
    "R18 baseline:",
    f"{base18:,}",
)

print(
    "R18 ProCBAM :",
    f"{pro18:,}",
)

print(
    "R18 added   :",
    pro18 - base18,
)

print()

print(
    "R34 baseline:",
    f"{base34:,}",
)

print(
    "R34 ProCBAM :",
    f"{pro34:,}",
)

print(
    "R34 added   :",
    pro34 - base34,
)

R18 baseline: 11,176,512
R18 ProCBAM : 11,355,472
R18 added   : 178960

R34 baseline: 21,284,672
R34 ProCBAM : 21,608,664
R34 added   : 323992


### Forward NWPU and AID

In [35]:
procbam18.eval()


for size in [
    256,
    600,
]:

    x = torch.randn(
        1,
        3,
        size,
        size,
    )

    with torch.no_grad():
        output = procbam18(x)

    print(
        f"{size}x{size}:"
    )

    print(
        " feature =",
        output[
            "features"
        ].shape,
    )

    print(
        " final fmap =",
        output[
            "fmaps"
        ][-1].shape,
    )

    assert (
        output[
            "features"
        ].shape
        == (1, 512)
    )


print(
    "ProCBAM forward: PASSED"
)

256x256:
 feature = torch.Size([1, 512])
 final fmap = torch.Size([1, 512, 8, 8])
600x600:
 feature = torch.Size([1, 512])
 final fmap = torch.Size([1, 512, 19, 19])
ProCBAM forward: PASSED


### Ensure ProCBAM gradient

In [36]:
procbam18 = resnet18_remote(
    pretrained=False,
    args=procbam_args,
)

procbam18.train()


x = torch.randn(
    1,
    3,
    256,
    256,
)

output = procbam18(x)

loss = (
    output[
        "features"
    ].mean()
)

procbam18.zero_grad()

loss.backward()


channel_grad = 0
spatial_grad = 0
se_grad = 0


for name, parameter in (
    procbam18.named_parameters()
):

    if parameter.grad is None:
        continue

    if (
        ".attention."
        "channel_attention."
        in name
    ):
        channel_grad += 1

    elif (
        ".attention."
        "spatial_attention."
        in name
    ):
        spatial_grad += 1

    elif (
        ".attention."
        "squeeze_excitation."
        in name
    ):
        se_grad += 1


print(
    "CAM params with gradient:",
    channel_grad,
)

print(
    "SAM params with gradient:",
    spatial_grad,
)

print(
    "SE params with gradient:",
    se_grad,
)


assert channel_grad > 0
assert spatial_grad > 0
assert se_grad > 0


print(
    "ProCBAM gradient flow: PASSED"
)

CAM params with gradient: 32
SAM params with gradient: 24
SE params with gradient: 32
ProCBAM gradient flow: PASSED


### Test IncrementalNet

In [37]:
inc_procbam_args = {
    "convnet_type":
        "remote_resnet18",

    "attention_type":
        "procbam",

    "procbam_reduction":
        16,

    "procbam_spatial_kernel":
        7,

    "procbam_se_reduction":
        16,
}


network = IncrementalNet(
    inc_procbam_args,
    pretrained=False,
)

network.update_fc(
    nb_classes=14
)

network.eval()


x = torch.randn(
    1,
    3,
    600,
    600,
)


with torch.no_grad():
    output = network(x)


print(
    "Feature:",
    output[
        "features"
    ].shape,
)

print(
    "Logits:",
    output[
        "logits"
    ].shape,
)


assert (
    output[
        "features"
    ].shape
    == (1, 512)
)

assert (
    output[
        "logits"
    ].shape
    == (1, 14)
)


print(
    "ProCBAM IncrementalNet: PASSED"
)

Feature: torch.Size([1, 512])
Logits: torch.Size([1, 14])
ProCBAM IncrementalNet: PASSED


### Check variants registry

In [38]:
from thesis.backbones.variants import (
    get_all_backbone_variants,
    get_backbone_variant,
)


variants = (
    get_all_backbone_variants()
)


print(
    "Total variants:",
    len(variants)
)

for name, config in (
    variants.items()
):
    print(
        name,
        "->",
        config[
            "convnet_type"
        ],
        "+",
        config[
            "attention_type"
        ],
    )


assert len(variants) == 8

print(
    "Backbone registry: PASSED"
)

Total variants: 8
r18_none -> remote_resnet18 + none
r18_eca -> remote_resnet18 + eca
r18_cbam -> remote_resnet18 + cbam
r18_procbam -> remote_resnet18 + procbam
r34_none -> remote_resnet34 + none
r34_eca -> remote_resnet34 + eca
r34_cbam -> remote_resnet34 + cbam
r34_procbam -> remote_resnet34 + procbam
Backbone registry: PASSED


### Ensuring registry safe from accidental mutation

In [40]:
config_a = get_backbone_variant(
    "r18_eca"
)

config_a[
    "eca_gamma"
] = 999


config_b = get_backbone_variant(
    "r18_eca"
)


print(
    "Stored gamma:",
    config_b[
        "eca_gamma"
    ]
)


assert (
    config_b[
        "eca_gamma"
    ]
    == 2.0
)

print(
    "Registry isolation: PASSED"
)

Stored gamma: 2.0
Registry isolation: PASSED


### Test PyCIL get_convnet()

In [41]:
from utils.inc_net import (
    get_convnet,
)


expected_params = {

    "r18_none":
        11_176_512,

    "r18_eca":
        11_176_548,

    "r18_cbam":
        11_266_392,

    "r18_procbam":
        11_355_472,

    "r34_none":
        21_284_672,

    "r34_eca":
        21_284_746,

    "r34_cbam":
        21_447_468,

    "r34_procbam":
        21_608_664,
}


def count_parameters(model):
    return sum(
        parameter.numel()
        for parameter
        in model.parameters()
    )


for variant_name, config in (
    variants.items()
):

    model = get_convnet(
        config,
        pretrained=False,
    )

    params = count_parameters(
        model
    )

    print(
        f"{variant_name:15s}",
        f"{params:,}",
    )

    assert (
        model.out_dim
        == 512
    )

    assert (
        params
        == expected_params[
            variant_name
        ]
    )


print(
    "get_convnet compatibility: PASSED"
)

r18_none        11,176,512
r18_eca         11,176,548
r18_cbam        11,266,392
r18_procbam     11,355,472
r34_none        21,284,672
r34_eca         21,284,746
r34_cbam        21,447,468
r34_procbam     21,608,664
get_convnet compatibility: PASSED


### Validate interface of all backbone

In [42]:
for variant_name, config in (
    variants.items()
):

    model = get_convnet(
        config,
        pretrained=False,
    )

    assert hasattr(
        model,
        "out_dim",
    )

    assert hasattr(
        model,
        "last_conv",
    )

    assert hasattr(
        model,
        "layer1",
    )

    assert hasattr(
        model,
        "layer2",
    )

    assert hasattr(
        model,
        "layer3",
    )

    assert hasattr(
        model,
        "layer4",
    )

    assert (
        model.out_dim
        == 512
    )

    print(
        variant_name,
        ": interface OK"
    )


print(
    "Backbone interface: PASSED"
)

r18_none : interface OK
r18_eca : interface OK
r18_cbam : interface OK
r18_procbam : interface OK
r34_none : interface OK
r34_eca : interface OK
r34_cbam : interface OK
r34_procbam : interface OK
Backbone interface: PASSED


### Test full foward compatibility matrix

In [43]:
import gc


input_specs = {

    "nwpu": {
        "size": 256,
        "final_fmap": (
            1,
            512,
            8,
            8,
        ),
    },

    "aid": {
        "size": 600,
        "final_fmap": (
            1,
            512,
            19,
            19,
        ),
    },
}


forward_results = []


for variant_name, config in (
    variants.items()
):

    model = get_convnet(
        config,
        pretrained=False,
    )

    model.eval()

    for dataset_name, spec in (
        input_specs.items()
    ):

        size = spec["size"]

        x = torch.randn(
            1,
            3,
            size,
            size,
        )

        with torch.inference_mode():
            output = model(x)

        feature_shape = tuple(
            output[
                "features"
            ].shape
        )

        final_fmap_shape = tuple(
            output[
                "fmaps"
            ][-1].shape
        )

        assert (
            feature_shape
            == (1, 512)
        )

        assert (
            final_fmap_shape
            == spec[
                "final_fmap"
            ]
        )

        assert torch.isfinite(
            output[
                "features"
            ]
        ).all()

        forward_results.append(
            {
                "variant":
                    variant_name,

                "dataset":
                    dataset_name,

                "feature_shape":
                    feature_shape,

                "final_fmap":
                    final_fmap_shape,

                "finite":
                    True,
            }
        )

        print(
            f"{variant_name:15s}"
            f" | {dataset_name:4s}"
            f" | feature="
            f"{feature_shape}"
            f" | fmap="
            f"{final_fmap_shape}"
        )

        del x
        del output

    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


assert (
    len(forward_results)
    == 16
)

print(
    "\nFull forward matrix: PASSED"
)

r18_none        | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r18_none        | aid  | feature=(1, 512) | fmap=(1, 512, 19, 19)
r18_eca         | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r18_eca         | aid  | feature=(1, 512) | fmap=(1, 512, 19, 19)
r18_cbam        | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r18_cbam        | aid  | feature=(1, 512) | fmap=(1, 512, 19, 19)
r18_procbam     | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r18_procbam     | aid  | feature=(1, 512) | fmap=(1, 512, 19, 19)
r34_none        | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r34_none        | aid  | feature=(1, 512) | fmap=(1, 512, 19, 19)
r34_eca         | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r34_eca         | aid  | feature=(1, 512) | fmap=(1, 512, 19, 19)
r34_cbam        | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r34_cbam        | aid  | feature=(1, 512) | fmap=(1, 512, 19, 19)
r34_procbam     | nwpu | feature=(1, 512) | fmap=(1, 512, 8, 8)
r34_procbam     | aid  | f

### Validate attention module count

In [44]:
from thesis.attention.eca import (
    EfficientChannelAttention,
)

from thesis.attention.cbam import (
    CBAM,
)

from thesis.attention.procbam import (
    ProCBAM,
)


expected_attention_counts = {

    "r18_none":
        ("none", 8),

    "r18_eca":
        ("eca", 8),

    "r18_cbam":
        ("cbam", 8),

    "r18_procbam":
        ("procbam", 8),

    "r34_none":
        ("none", 16),

    "r34_eca":
        ("eca", 16),

    "r34_cbam":
        ("cbam", 16),

    "r34_procbam":
        ("procbam", 16),
}


for variant_name, config in (
    variants.items()
):

    model = get_convnet(
        config,
        pretrained=False,
    )

    attention_type, expected_count = (
        expected_attention_counts[
            variant_name
        ]
    )

    identities = sum(
        isinstance(
            module,
            nn.Identity,
        )
        for module
        in model.modules()
    )

    ecas = sum(
        isinstance(
            module,
            EfficientChannelAttention,
        )
        for module
        in model.modules()
    )

    cbams = sum(
        isinstance(
            module,
            CBAM,
        )
        for module
        in model.modules()
    )

    procbams = sum(
        isinstance(
            module,
            ProCBAM,
        )
        for module
        in model.modules()
    )


    if attention_type == "none":

        assert identities == expected_count
        assert ecas == 0
        assert cbams == 0
        assert procbams == 0

    elif attention_type == "eca":

        assert ecas == expected_count
        assert cbams == 0
        assert procbams == 0

    elif attention_type == "cbam":

        assert ecas == 0
        assert cbams == expected_count
        assert procbams == 0

    elif attention_type == "procbam":

        assert ecas == 0
        assert cbams == 0
        assert procbams == expected_count


    print(
        variant_name,
        ": PASSED"
    )


print(
    "Attention isolation: PASSED"
)

r18_none : PASSED
r18_eca : PASSED
r18_cbam : PASSED
r18_procbam : PASSED
r34_none : PASSED
r34_eca : PASSED
r34_cbam : PASSED
r34_procbam : PASSED
Attention isolation: PASSED


### Test all variant through IncrementalNet

In [45]:
from utils.inc_net import (
    IncrementalNet,
)
incremental_results = []

for variant_name, config in (
    variants.items()
):

    network = IncrementalNet(
        config,
        pretrained=False,
    )

    network.update_fc(
        nb_classes=14
    )

    network.eval()


    x = torch.randn(
        1,
        3,
        256,
        256,
    )


    with torch.inference_mode():
        output = network(x)


    assert (
        output[
            "features"
        ].shape
        == (1, 512)
    )

    assert (
        output[
            "logits"
        ].shape
        == (1, 14)
    )

    assert len(
        output["fmaps"]
    ) == 4


    incremental_results.append(
        {
            "variant":
                variant_name,

            "feature_dim":
                output[
                    "features"
                ].shape[1],

            "logit_dim":
                output[
                    "logits"
                ].shape[1],
        }
    )


    print(
        f"{variant_name:15s}"
        " -> PASSED"
    )


    del network
    del x
    del output

    gc.collect()


print(
    "\nAll IncrementalNet variants: PASSED"
)

r18_none        -> PASSED
r18_eca         -> PASSED
r18_cbam        -> PASSED
r18_procbam     -> PASSED
r34_none        -> PASSED
r34_eca         -> PASSED
r34_cbam        -> PASSED
r34_procbam     -> PASSED

All IncrementalNet variants: PASSED


### Test dynamic classifier expansion

In [46]:
config = get_backbone_variant(
    "r18_procbam"
)


network = IncrementalNet(
    config,
    pretrained=False,
)


network.update_fc(
    nb_classes=14
)

old_weights = (
    network.fc.weight
    .detach()
    .clone()
)


network.update_fc(
    nb_classes=18
)


print(
    "New FC shape:",
    network.fc.weight.shape,
)


assert (
    network.fc.weight.shape[0]
    == 18
)

assert torch.equal(
    network.fc.weight[
        :14
    ],
    old_weights,
)


print(
    "Old classifier weights "
    "preserved: PASSED"
)

print(
    "Incremental classifier "
    "expansion: PASSED"
)

New FC shape: torch.Size([18, 512])
Old classifier weights preserved: PASSED
Incremental classifier expansion: PASSED


### Test frozen-backbone compatibility

In [47]:
attention_variants = [
    "r18_eca",
    "r18_cbam",
    "r18_procbam",
]


for variant_name in (
    attention_variants
):

    config = get_backbone_variant(
        variant_name
    )

    network = IncrementalNet(
        config,
        pretrained=False,
    )

    network.update_fc(
        14
    )


    for parameter in (
        network
        .convnet
        .parameters()
    ):
        parameter.requires_grad = False


    convnet_trainable = [
        name
        for name, parameter
        in network
        .convnet
        .named_parameters()
        if parameter.requires_grad
    ]


    fc_trainable = [
        name
        for name, parameter
        in network
        .fc
        .named_parameters()
        if parameter.requires_grad
    ]


    print(
        variant_name
    )

    print(
        "  convnet trainable:",
        len(
            convnet_trainable
        )
    )

    print(
        "  fc trainable:",
        len(
            fc_trainable
        )
    )


    assert (
        len(
            convnet_trainable
        )
        == 0
    )

    assert (
        len(
            fc_trainable
        )
        > 0
    )


print(
    "Frozen-backbone compatibility: "
    "PASSED"
)

r18_eca
  convnet trainable: 0
  fc trainable: 2
r18_cbam
  convnet trainable: 0
  fc trainable: 2
r18_procbam
  convnet trainable: 0
  fc trainable: 2
Frozen-backbone compatibility: PASSED


### Checkpoint compatibility

In [48]:
import copy


for variant_name, config in (
    variants.items()
):

    model_a = get_convnet(
        config,
        pretrained=False,
    )

    state_dict = copy.deepcopy(
        model_a.state_dict()
    )


    model_b = get_convnet(
        config,
        pretrained=False,
    )


    load_result = (
        model_b.load_state_dict(
            state_dict,
            strict=True,
        )
    )


    assert (
        len(
            load_result.missing_keys
        )
        == 0
    )

    assert (
        len(
            load_result.unexpected_keys
        )
        == 0
    )


    print(
        variant_name,
        ": checkpoint OK"
    )


print(
    "State-dict compatibility: PASSED"
)

r18_none : checkpoint OK
r18_eca : checkpoint OK
r18_cbam : checkpoint OK
r18_procbam : checkpoint OK
r34_none : checkpoint OK
r34_eca : checkpoint OK
r34_cbam : checkpoint OK
r34_procbam : checkpoint OK
State-dict compatibility: PASSED


### Conclusion with DF

In [49]:
import pandas as pd


rows = []


for variant_name, config in (
    variants.items()
):

    model = get_convnet(
        config,
        pretrained=False,
    )

    rows.append(
        {
            "variant":
                variant_name,

            "backbone":
                config[
                    "convnet_type"
                ],

            "attention":
                config[
                    "attention_type"
                ],

            "parameters":
                count_parameters(
                    model
                ),

            "feature_dim":
                model.out_dim,
        }
    )


backbone_summary = (
    pd.DataFrame(rows)
)


display(
    backbone_summary
)

,variant,backbone,attention,parameters,feature_dim
0,r18_none,remote_resnet18,none,11176512,512
1,r18_eca,remote_resnet18,eca,11176548,512
2,r18_cbam,remote_resnet18,cbam,11266392,512
3,r18_procbam,remote_resnet18,procbam,11355472,512
4,r34_none,remote_resnet34,none,21284672,512
5,r34_eca,remote_resnet34,eca,21284746,512
6,r34_cbam,remote_resnet34,cbam,21447468,512
7,r34_procbam,remote_resnet34,procbam,21608664,512
